# FitRec-Attn no Google Colab

**Antes de começar:**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.
2. No seu Google Drive, crie a pasta `fitrec-data` e coloque nela o arquivo `fitrec_compact.npz` (gerado com `python compact_data.py` no PC de 32 GB).

Os modelos salvos vão para `fitrec-data/fitrec-attn/model_states/` no Drive. Se o Colab desconectar, mude `NOVA_EXECUCAO` para `False` na última célula e rode todas de novo: o treino continua da última época concluída.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/fitrec-data

In [ ]:
%cd /content
![ -d fit-rec ] || git clone -b fitrec-attn-run https://github.com/laracmv/fit-rec.git
%cd /content/fit-rec/FitRec-Attn
!git pull
!rm -rf data && ln -s /content/drive/MyDrive/fitrec-data data
!pip install -q haversine
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Treino completo (até 50 épocas, para sozinho após 3 sem melhora).
# NOVA_EXECUCAO = True  -> começa um treino do zero, numa pasta nova.
# NOVA_EXECUCAO = False -> continua a execução mais recente (use depois de uma desconexão;
#                          as opções do experimento são lidas da própria execução).
# A cada época o histórico de loss (treino e validação) é salvo em history.json, dentro da pasta da execução.
NOVA_EXECUCAO = True

# Experimento (só vale para NOVA_EXECUCAO = True). Deixe "" para o modelo original.
#   "--lr-decay 0.7"  taxa de aprendizado cai 30% a cada época
#   "--no-context"    sem o treino anterior (ablação)
#   "--no-user"       sem o embedding de usuário (ablação)
#   "--sport"         inclui o esporte como entrada
EXPERIMENTO = ""

%cd /content/fit-rec/FitRec-Attn
if NOVA_EXECUCAO:
    !python -u fitrect-attn.py {EXPERIMENTO} 2>&1 | tee -a data/treino_colab.log
else:
    !python -u fitrect-attn.py --resume 2>&1 | tee -a data/treino_colab.log